## Fine-tuning a Frontier Model

In [1]:
# libraries
import os
import re
import json
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items import Item
from pricer.evaluator import evaluate

In [2]:
# setting the environment
LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']

login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
# loading the dataset
username = 'ed-donner'
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

# retrieving the train, test and validation data
train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,}")

Loaded 800,000 training items, 10,000 validation items, 10,000


In [4]:
openai = OpenAI()

### Data Size

OpenAI recommends fine-tuning with a small population of 50-100 examples

In [6]:
fine_tune_train = train[:100]
fine_tune_validation = val[:50]

### Step 1
Preparing the data for fine-tuning in JSONL format and upload to OpenAI

In [7]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"

    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.0f}.00"}
    ]

In [10]:
messages_for(fine_tune_train[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4" minimum center‑to‑center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation.'},
 {'role': 'assistant', 'content': '$64.00'}]

In [21]:
def make_jsonl(items):
    result = ""

    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += messages_str + '\n'
        
    return result.strip()

In [22]:
print(make_jsonl(train[:3]))

[{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single\u2011piece oil\u2011rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4\" minimum center\u2011to\u2011center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation."}, {"role": "assistant", "content": "$64.00"}]
[{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Mini Electric Air Duster Fan  \nCategory: Electronics  \nBrand: Kica  \nDescription: Ultra\u2011compact 86,000\u202fRPM electric air duster with 11\u202fm/s wind speed for precise cleaning and inflation.  \nDetails: Powered by a 9.99\u202fWh motor, adjustable in four speed levels, it uses three lithium\u2011ion batteries 

In [23]:
# converting the items into jsonl and write to a file

def write_jsonl(items, filename):
    with open(filename, 'w') as file:
        jsonl = make_jsonl(items)
        file.write(jsonl)

In [24]:
write_jsonl(fine_tune_train, 'jsonl/fine_tune_train.jsonl')

In [25]:
write_jsonl(fine_tune_validation, 'jsonl/fine_tune_validation.jsonl')

In [26]:
with open('jsonl/fine_tune_train.jsonl', 'rb') as f:
    train_file = openai.files.create(file=f, purpose='fine-tune')

In [27]:
train_file

FileObject(id='file-Be1PpRtxcupy7dCMh9fuVV', bytes=53721, created_at=1792173131, filename='fine_tune_train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [28]:
with open('jsonl/fine_tune_validation.jsonl', 'rb') as f:
    validation_file = openai.files.create(file=f, purpose='fine-tune')

In [29]:
validation_file

FileObject(id='file-16YLoUy2ftvUfEvjH3255p', bytes=26937, created_at=1792173299, filename='fine_tune_validation.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

### Fine-tuning

In [ ]:
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model='gpt-4.1-nano-2025-04-14',
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 16},
    suffix='pricer'
)

In [ ]:
openai.fine_tuning.jobs.list(limit=1)

In [ ]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id

In [ ]:
openai.fine_tuning.jobs.retrieve(job_id)

### Testing the Fine-tuned Model

In [ ]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [ ]:
fine_tuned_model_name

In [ ]:
def test_messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"

    return [
        {"role": "user", "content": message}
    ]

In [ ]:
test_messages_for(test[0])

In [ ]:
def gpt_4_1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        seed=42,
        max_tokens=7
    )

    return response.choices[0].message.content

In [ ]:
print(test[0].price)
print(gpt_4_1_nano_fine_tuned(test[0]))

In [ ]:
evaluate(gpt_4_1_nano_fine_tuned, test)

## Using Deep Neural Network

In [32]:
from dotenv import load_dotenv
import os
from huggingface_hub import login
from pricer.evaluator import evaluate
from pricer.deep_neural_network import DeepNeuralNetworkRunner
from pricer.items import Item

In [35]:
# setting up the environment
LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']

login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [36]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


In [37]:
runner = DeepNeuralNetworkRunner(train=train, val=val[:1000])

runner.setup()

Deep Neural Network created with 289,128,449 parameters
Using mps


In [ ]:
runner.train(epochs=5)
runner.save('deep_neural_network.pth')

In [40]:
runner.load('deep_neural_network.pth')

In [41]:
def deep_neural_network(item):
    return runner.inference(item)

evaluate(deep_neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$19 $40 $15 $21 $11 $88 $46 $14 $8 $5 $97 $49 $16 $2 $20 $1 $39 $13 $15 $241 $32 $24 $11 $117 $55 $229 $184 $15 $156 $50 $36 $19 $21 $50 $7 $162 $24 $22 $46 $2 $91 $28 $19 $37 $112 $4 $12 $1 $70 $2 $8 $37 $255 $57 $24 $42 $24 $26 $85 $20 $173 $43 $35 $35 $391 $26 $21 $318 $18 $82 $17 $6 $46 $41 $16 $3 $24 $1 $0 $3 $94 $25 $2 $53 $14 $28 $123 $34 $52 $121 $5 $8 $0 $3 $13 $14 $2 $36 $26 $227 $3 $45 $2 $82 $10 $26 $17 $250 $13 $101 $29 $15 $4 $38 $46 $48 $26 $14 $33 $112 $6 $51 $4 $27 $58 $53 $2 $35 $9 $12 $19 $17 $21 $1 $53 $3 $16 $34 $74 $22 $27 $174 $44 $43 $4 $11 $0 $237 $72 $12 $0 $96 $6 $42 $6 $152 $57 $8 $5 $17 $34 $11 $4 $0 $232 $1 $207 $14 $21 $11 $49 $5 $378 $20 $39 $9 $6 $17 $30 $31 $124 $2 $138 $14 $3 $24 $83 $12 $21 $15 $4 $8 $1 $45 $9 $41 $10 $3 $21 $12 